# Task 1b — 低频的差距是不是因为仿真里没有躯干？

**2026-10-11 建。** Task 1 发现：0.2–5 kHz 里，几何能预测仿真（SYMARE DTF R² 0.27），却几乎预测不了实测（0.05）；
这个人自己的仿真也只能解释实测的一小部分。Task 1 用的两套仿真都没有躯干：SYMARE `HE16kHz` 是"头 + 耳"网格，
SONICOM 的网格到脖子为止；实测里有躯干和肩膀。SYMARE 另有一套 `HTE05kHz`：同一个 FM-BEM，"头 + 躯干 + 耳"网格，
约 5 kHz 以下有效（BEM-ready 网格满足 5 kHz 下最长边 < λ/6，所以 5 kHz 以下两套网格的精度都够，差别主要是躯干）。

**问题**：把躯干加进仿真，仿真对这个人实测的预测会不会明显变好？

## 设计（跑之前写死）

- **受试者 / 折**：SYMARE 46 个开发集受试者（AE train + val），折与 Task 1 相同（外 10 / 内 5，seed 0）。锁定的 15 人不读。
- **频段**：0.2–5 kHz（bin 2–26，375–4875 Hz，与 Task 1 的频段相同），在 HTE 的有效范围内（HTE 5.3 kHz 以上衰减）。
- **目标**：实测 DTF（dB，仰角 ≥ −45° 的 2652 个方向，每只耳每个 bin 减去自己的方向平均）。不经过 AE：AE 没见过 HTE，HTE 高频也无效。
- **主检验（校准）**：这个人自己的仿真 DTF → 实测 DTF，核 ridge（与 Task 1 相同的引擎），HTE 和 HE 各跑一次；
  差值 R²(HTE → 实测) − R²(HE → 实测) 按人配对 bootstrap 1000 次的 95% CI。
  - CI 在 0 以上：加了躯干的仿真更能预测实测 → 缺躯干是低频差距的一部分原因。
  - CI 含 0 或在 0 以下：没有证据表明缺躯干能解释低频差距；剩下的候选在测量一侧（低频的均衡/补偿、佩戴位置）。
- **次要（描述）**：每只耳仿真偏差与实测偏差的 Pearson 相关 ρ（偏差 = 减去同侧 46 人平均），HE 与 HTE 配对比较；
  几何（Task 1 的 19 个 landmark：耳廓、中矢面和一只眼，没有躯干上的点）→ HE / HTE / 实测；每 bin 的 R² 曲线；HE 和 HTE 的校准各做 100 次按人置换。

## Drive 上要有的文件

`ELEC5021/data_48k_2026-10-10.zip`（Task 1 用的那份，只解压 SYMARE 两个文件夹）、`ELEC5021/data_48k_hte_2026-10-11.zip`（新，10-11 打包，
`preprocess_48k/pack_hte.py` 已检查：ID、网格、配对、逐数组 SHA）、`ELEC5021/split_48k.json`、`ELEC5021/SYMARE/3DShapeData.mat`。
结果写到 `HRTF_Checkpoints/task1_v1/`。**CPU 运行时即可**，从上到下跑，几分钟。

In [ ]:
!pip install healpy -q

In [ ]:
# ==========================================================================
# Cell 0 — 路径 · 数据（两个 zip 都按 SHA-256 核对后解压到本地盘；读到的每个数组再按 manifest 核对）
# ==========================================================================
import os, json, time, hashlib, zipfile, shutil
import numpy as np
import healpy as hp
import scipy.io as sio
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
PATHS = dict(
    data_zip  = f'{DRIVE}/ELEC5021/data_48k_2026-10-10.zip',
    hte_zip   = f'{DRIVE}/ELEC5021/data_48k_hte_2026-10-11.zip',
    split     = f'{DRIVE}/ELEC5021/split_48k.json',
    landmarks = f'{DRIVE}/ELEC5021/SYMARE/3DShapeData.mat',
)
for k, p in PATHS.items():
    print(f'{k:10s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), '先把缺失的文件放好'
ZIP_SHA256, SPLIT_SHA256, HTE_SHA256 = 'f51c6b072123e70d6d8a0691373ab90e3f6db9856ce6306f65d949c87fbf97c7', 'dc89bc454fef351951283cccb3772fdd3e552bbdde5d13668f8f576f4cd83561', '0b0f9ab81e0a2e985294730f30e71ade20aadb7b83db394e00676e50fd617380'
OUT = f'{DRIVE}/HRTF_Checkpoints/task1_v1'
os.makedirs(OUT, exist_ok=True)


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def stage(zpath, digest, dest, keep=lambda n: True):
    stamp = f'{dest}/.zip_ok_{digest[:16]}'
    if not os.path.exists(stamp):
        t0 = time.time()
        assert sha256(zpath) == digest, f'{os.path.basename(zpath)} 的 SHA-256 不符'
        shutil.rmtree(dest, ignore_errors=True); os.makedirs(dest)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(dest, [n for n in z.namelist() if keep(n)])
        open(stamp, 'w').write(digest)
        print(f'{os.path.basename(zpath)} 核对并解压 ({time.time() - t0:.0f}s)')


# 主数据只解压 SYMARE 实测、SYMARE_BEM（头 + 耳）和镜像表
stage(PATHS['data_zip'], ZIP_SHA256, '/content/task1b_data',
      lambda n: n.startswith(('data_48k/SYMARE_48k/', 'data_48k/SYMARE_BEM_48k/')) or n == 'data_48k/healpix_mirror_idx.npy')
stage(PATHS['hte_zip'], HTE_SHA256, '/content/task1b_hte')
DATA = '/content/task1b_data/data_48k'
DATA_HTE = '/content/task1b_hte/data_48k_hte'
assert sha256(PATHS['split']) == SPLIT_SHA256, 'split_48k.json 的 SHA-256 不符'
SPLIT = json.load(open(PATHS['split']))
MIRROR = np.load(f'{DATA}/healpix_mirror_idx.npy')
assert np.array_equal(MIRROR[MIRROR], np.arange(3072))
meta = json.load(open(f'{DATA_HTE}/SYMARE_HTE_48k/meta.json'))
ver = json.load(open(f'{DATA_HTE}/SYMARE_HTE_48k/verification_hte.json'))
assert meta['git']['preprocess_48k_uncommitted'] is False and meta['ild_band_hz'] == [1500.0, 5000.0]
print(f"SYMARE_HTE：preprocess commit {meta['git']['commit'][:7]} | 打包检查：HTE 与自己的 HE 偏差相关 "
      f"{ver['left']['deviation_corr_own_median']:.2f} / {ver['right']['deviation_corr_own_median']:.2f}（左 / 右），与别人 ≈ 0")

In [ ]:
# ==========================================================================
# Cell 1 — 设定（跑之前写死，不要改）
# ==========================================================================
FS = 48000
FREQS = np.arange(129) * FS / 256
BAND_BINS = np.flatnonzero((FREQS >= 200) & (FREQS <= 5000))           # bin 2–26
assert BAND_BINS[0] == 2 and BAND_BINS[-1] == 26
ALPHA_REL = list(np.logspace(-4, 5, 37))
N_FOLDS, N_INNER, SEED = 10, 5, 0
N_PERM, N_BOOT = 100, 1000
EL = 90 - np.degrees(hp.pix2ang(16, np.arange(3072))[0])
MEAS_DIRS = EL >= -45.0
print(f'{len(BAND_BINS)} 个 bin（{FREQS[BAND_BINS[0]]:.1f}–{FREQS[BAND_BINS[-1]]:.1f} Hz）× {MEAS_DIRS.sum()} 个方向')

In [ ]:
# ==========================================================================
# Cell 2 — 回归引擎：核 ridge 的对偶形式（与 FFM_3_2_1 Cell 6 的预测完全相同）
#
# FFM_3_2_1 在每折内对目标做 PCA 只是为了快；这里直接用目标的 Gram 矩阵 Y Yᵀ 算误差，
# 对任意维度的目标（latent 5504 维、DTF 约 22 万维）都一样快。
# ==========================================================================
def subject_folds(subjects, k, rng):
    p = rng.permutation(subjects)
    return [np.sort(a) for a in np.array_split(p, min(k, len(p)))]


def make_plan(ns, seed=SEED):
    # FFM_3_2_1 的 build_plan：随机数的调用顺序相同，所以折相同
    rng = np.random.default_rng(seed)
    subj = np.repeat(np.arange(ns), 2)
    rows = lambda s: np.flatnonzero(np.isin(subj, s))
    plan = []
    for te in subject_folds(np.arange(ns), N_FOLDS, rng):
        tr = np.setdiff1d(np.arange(ns), te)
        inner = [(rows(np.setdiff1d(tr, v)), rows(v)) for v in subject_folds(tr, N_INNER, rng)]
        plan.append(((rows(tr), rows(te)), inner))
    return plan


def centred_pair(G, tr, va):
    Gtt, Gvt = G[np.ix_(tr, tr)], G[np.ix_(va, tr)]
    cm, tot = Gtt.mean(0), Gtt.mean()
    return Gtt - cm[None] - cm[:, None] + tot, Gvt - cm[None] - Gvt.mean(1)[:, None] + tot


def centred_vv_diag(G, tr, va):
    r = G[np.ix_(va, tr)].mean(1)
    return np.diag(G[np.ix_(va, va)]) - 2 * r + G[np.ix_(tr, tr)].mean()


def weights(K, tr, va, alphas):
    Ktt, Kvt = centred_pair(K, tr, va)
    lam, Q = np.linalg.eigh(Ktt)
    lam = np.clip(lam, 0, None)
    KvQ, scale = Kvt @ Q, np.trace(Ktt) / len(tr)
    return [(KvQ / (lam + a * scale)) @ Q.T for a in alphas]


def row_sse(W, G, tr, va):
    CYtt, CYvt = centred_pair(G, tr, va)
    return centred_vv_diag(G, tr, va) - 2 * np.einsum('ij,ij->i', W, CYvt) + np.einsum('ij,ij->i', W @ CYtt, W)


def run_cv(K, T, plan, keep_pred=False, parts=True):
    # T: {'G': Y Yᵀ, 'parts': {名字: 部分目标的 Gram}, 'Y': 显式目标（要预测值时）}
    n = K.shape[0]
    sse, sst = np.zeros(n), np.zeros(n)
    psse = {k: np.zeros(n) for k in T['parts']} if parts else {}
    psst = {k: np.zeros(n) for k in T['parts']} if parts else {}
    pred = np.zeros_like(T['Y']) if keep_pred else None
    chosen = []
    for (tr, va), inner in plan:
        tot = np.zeros(len(ALPHA_REL))
        for itr, iva in inner:
            for j, W in enumerate(weights(K, itr, iva, ALPHA_REL)):
                tot[j] += row_sse(W, T['G'], itr, iva).sum()
        j = int(np.argmin(tot)); chosen.append(j)
        W = weights(K, tr, va, [ALPHA_REL[j]])[0]
        sse[va] = row_sse(W, T['G'], tr, va); sst[va] = centred_vv_diag(T['G'], tr, va)
        for k, G in (T['parts'].items() if parts else ()):
            psse[k][va] = row_sse(W, G, tr, va); psst[k][va] = centred_vv_diag(G, tr, va)
        if keep_pred:
            Ytr = T['Y'][tr]; my = Ytr.mean(0)
            pred[va] = my + W @ (Ytr - my)
    return dict(r2=float(1 - sse.sum() / sst.sum()), sse=sse, sst=sst, alpha_idx=chosen, pred=pred,
                parts={k: float(1 - psse[k].sum() / psst[k].sum()) for k in psse})


gram = lambda Y: Y @ Y.T


def zscore(X):
    sd = X.std(0)
    keep = sd > 1e-12 * max(sd.max(), 1e-300)
    return (X[:, keep] - X[:, keep].mean(0)) / sd[keep]


def unit_trace(G):
    n = len(G); C = np.eye(n) - 1.0 / n
    return G / (np.trace(C @ G @ C) / n)


def permuted(G, perm):
    r = np.array([2 * perm[k] + s for k in range(len(perm)) for s in (0, 1)])
    return G[np.ix_(r, r)]


# 自检：随机特征的 CV-R² 应在 0 附近或为负
_plan = make_plan(40); _Y = np.random.default_rng(3).standard_normal((80, 50)); _X = np.random.default_rng(4).standard_normal((80, 30))
print(f"自检：随机特征 CV-R² = {run_cv(gram(_X), dict(G=gram(_Y), parts={}), _plan, parts=False)['r2']:+.4f}（应 ≤ ~0）")

In [ ]:
# ==========================================================================
# Cell 3 — 数据 · 目标 · 几何（几何与 Task 1 Cell 5 的 symare_geometry 逐字相同）
# ==========================================================================
def load_rows(root, tag, ids):
    # 每人两行（左耳, 镜像后的右耳），只取频段内的 bin；每个数组按 manifest 核对 SHA-256
    d = f'{root}/{tag}_48k'
    man = {m['id']: m for m in json.load(open(f'{d}/manifest.json'))}
    X = np.empty((2 * len(ids), len(BAND_BINS), 3072), np.float64)
    for k, s in enumerate(ids):
        for side in ('left', 'right'):
            f = f'{d}/npy_{side}/{man[s][side]}'
            assert sha256(f) == man[s][f'{side}_sha256'], f
            a = np.load(f)[BAND_BINS]
            X[2 * k + (side == 'right')] = a[:, MIRROR] if side == 'right' else a
    return X


def dtf(X):
    # 方向 ≥ −45°，每只耳每个 bin 减去自己的方向平均
    D = X[:, :, MEAS_DIRS]
    return D - D.mean(2, keepdims=True)


def dtf_target(D):
    per_bin = np.einsum('nbd,mbd->bnm', D, D)
    parts = {f'bin{BAND_BINS[k]}': per_bin[k] for k in range(len(BAND_BINS))}
    return dict(G=per_bin.sum(0), parts=parts)


def symare_geometry(dev):
    # FFM_2_8：同侧 13 个耳廓点 + 5 个中矢面点 + 同侧眼点（31/32 是左右眼，不是肩），按耳构型（右耳=整体镜像）
    mat = sio.loadmat(PATHS['landmarks'], struct_as_record=False, squeeze_me=True)
    LM = np.stack(list(mat['A'])).astype(np.float64)
    assert LM.shape == (61, 33, 3) and not np.isnan(LM).any()
    EAR_P, MIDSAG, EAR_N, PAIR_N, PAIR_P = list(range(13)), list(range(13, 18)), list(range(18, 31)), 31, 32
    IPSI = EAR_P + MIDSAG + [PAIR_P]

    def mirror_config(P):
        Q = P.copy(); Q[:, 1] *= -1
        Q[EAR_P], Q[EAR_N] = P[EAR_N] * [1, -1, 1], P[EAR_P] * [1, -1, 1]
        Q[PAIR_P], Q[PAIR_N] = P[PAIR_N] * [1, -1, 1], P[PAIR_P] * [1, -1, 1]
        return Q

    rows = []
    for s in dev:
        P0 = LM[int(s[-2:]) - 1]                 # SYM_HAxx -> 第 xx-1 行；+y 是左耳（FFM_2_8 Cell 3b 声学判定）
        for c in (P0, mirror_config(P0)):
            c = c[IPSI]
            rows.append((c - c.mean(0)).ravel())
    X = zscore(np.array(rows))
    return X @ X.T / X.shape[1]

In [ ]:
# ==========================================================================
# Cell 4 — 分析
# ==========================================================================
DEV = sorted(SPLIT['SYMARE']['train'] + SPLIT['SYMARE']['val'])
assert len(DEV) == 46 and not set(DEV) & set(SPLIT['SYMARE']['test'])
PLAN = make_plan(len(DEV))
RNG = np.random.default_rng(SEED + 2)
D = {k: dtf(load_rows(r, t, DEV)) for k, (r, t) in
     {'meas': (DATA, 'SYMARE'), 'HE': (DATA, 'SYMARE_BEM'), 'HTE': (DATA_HTE, 'SYMARE_HTE')}.items()}
diff = D['HTE'] - D['HE']
print(f"46 人：HTE − HE 的 DTF 差 rms {np.sqrt((diff ** 2).mean()):.2f} dB（每 bin 最大 {np.sqrt((diff ** 2).mean((0, 2))).max():.2f} dB）")

T = {k: dtf_target(v) for k, v in D.items()}
K = {'HE': unit_trace(T['HE']['G']), 'HTE': unit_trace(T['HTE']['G']), 'geometry': unit_trace(symare_geometry(DEV))}
runs = {'HE -> meas': ('HE', 'meas'), 'HTE -> meas': ('HTE', 'meas'), 'geometry -> meas': ('geometry', 'meas'),
        'geometry -> HE': ('geometry', 'HE'), 'geometry -> HTE': ('geometry', 'HTE')}
R = {}
for label, (k, t) in runs.items():
    R[label] = run_cv(K[k], T[t], PLAN)
    edge = sum(c in (0, len(ALPHA_REL) - 1) for c in R[label]['alpha_idx'])
    print(f'  {label:<18} CV-R² {R[label]["r2"]:+.4f}' + (f'  （{edge} 折的 α 在网格边上）' if edge else ''))

perm = {}
for label in ('HE -> meas', 'HTE -> meas'):
    k, t = runs[label]
    null = np.array([run_cv(permuted(K[k], RNG.permutation(len(DEV))), T[t], PLAN, parts=False)['r2'] for _ in range(N_PERM)])
    perm[label] = dict(p=float((1 + (null >= R[label]['r2']).sum()) / (1 + N_PERM)), null_p95=float(np.percentile(null, 95)))
    print(f'  置换 {label:<12} p = {perm[label]["p"]:.3f}（零分布 95 分位 {perm[label]["null_p95"]:+.4f}）')

# ρ：每只耳，仿真偏差与实测偏差（各减去同侧 46 人平均）的 Pearson 相关
def rho(Dsim, Dmeas):
    out = np.empty(len(Dsim))
    for side in (0, 1):
        r = np.arange(side, len(Dsim), 2)
        a = (Dsim[r] - Dsim[r].mean(0)).reshape(len(r), -1); b = (Dmeas[r] - Dmeas[r].mean(0)).reshape(len(r), -1)
        a -= a.mean(1, keepdims=True); b -= b.mean(1, keepdims=True)
        out[r] = (a * b).sum(1) / np.sqrt((a * a).sum(1) * (b * b).sum(1))
    return out

RHO = {k: rho(D[k], D['meas']) for k in ('HE', 'HTE')}
print(f"  ρ（每只耳平均）：HE {RHO['HE'].mean():.3f}，HTE {RHO['HTE'].mean():.3f}")

per_subject = lambda v: v.reshape(-1, 2).sum(1)
r2_of = lambda lbl, i: 1 - per_subject(R[lbl]['sse'])[i].sum() / per_subject(R[lbl]['sst'])[i].sum()
rho_s = {k: v.reshape(-1, 2).mean(1) for k, v in RHO.items()}
stats = {
    'R2 meas: HTE - HE (primary)': lambda i: r2_of('HTE -> meas', i) - r2_of('HE -> meas', i),
    'rho: HTE - HE': lambda i: rho_s['HTE'][i].mean() - rho_s['HE'][i].mean(),
    'R2 geometry: HTE - HE': lambda i: r2_of('geometry -> HTE', i) - r2_of('geometry -> HE', i),
}
ns, CI = len(DEV), {}
for k, f in stats.items():
    bs = np.array([f(RNG.integers(0, ns, ns)) for _ in range(N_BOOT)])
    CI[k] = dict(obs=float(f(np.arange(ns))), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)))
    print(f'  bootstrap {k:<28} {CI[k]["obs"]:+.4f}  95% CI [{CI[k]["lo"]:+.4f}, {CI[k]["hi"]:+.4f}]')
p = CI['R2 meas: HTE - HE (primary)']
print('\n结论（按跑之前写死的规则）：' + ('加了躯干的仿真更能预测实测 → 缺躯干是低频差距的一部分原因' if p['lo'] > 0 else
      '没有证据表明缺躯干能解释低频差距（CI 不在 0 以上）'))

In [ ]:
# ==========================================================================
# Cell 5 — 保存 · 图
# ==========================================================================
import matplotlib.pyplot as plt
rho_bin = {k: [float(rho(D[k][:, [b]], D['meas'][:, [b]]).mean()) for b in range(len(BAND_BINS))] for k in ('HE', 'HTE')}
RES = dict(created=time.strftime('%Y-%m-%d %H:%M:%S'), notebook='task1/Task1b_TorsoCheck.ipynb',
           hte_zip_sha256=HTE_SHA256, n_subjects=ns, band_hz=[float(FREQS[BAND_BINS[0]]), float(FREQS[BAND_BINS[-1]])],
           cv={l: dict(r2=r['r2'], alpha_idx=r['alpha_idx']) for l, r in R.items()}, permutation=perm,
           rho_mean={k: float(v.mean()) for k, v in RHO.items()}, bootstrap=CI,
           per_bin={l: [R[l]['parts'][f'bin{b}'] for b in BAND_BINS] for l in R}, rho_per_bin=rho_bin,
           freqs_hz=[float(f) for f in FREQS[BAND_BINS]])
json.dump(RES, open(f'{OUT}/task1b_torso.json', 'w'), indent=1)

f = FREQS[BAND_BINS] / 1000
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for l, c in (('HE -> meas', 'C0'), ('HTE -> meas', 'C3'), ('geometry -> meas', 'k')):
    ax[0].plot(f, RES['per_bin'][l], c=c, lw=1.6, label=l)
for l, c in (('geometry -> HE', 'C0'), ('geometry -> HTE', 'C3'), ('geometry -> meas', 'k')):
    ax[1].plot(f, RES['per_bin'][l], c=c, lw=1.6, label=l)
for k, c in (('HE', 'C0'), ('HTE', 'C3')):
    ax[2].plot(f, rho_bin[k], c=c, lw=1.6, label=f'ρ({k}, measured)')
for a, t in zip(ax, ('Own simulation -> measured DTF (CV-R²)', 'Geometry -> each target (CV-R²)', 'Per-ear correlation of deviations')):
    a.axhline(0, c='k', lw=.6); a.grid(alpha=.3); a.legend(fontsize=8); a.set_title(f'SYMARE (46): {t}', fontsize=10); a.set_xlabel('kHz')
plt.tight_layout(); plt.savefig(f'{OUT}/task1b_torso.png', dpi=140); plt.show()
print(f'结果 -> {OUT}/task1b_torso.json, task1b_torso.png')